## Set base directory and path for VD, dissonance files

In [1]:
import pandas as pd
from pathlib import Path

BASE = Path("/home/patsakornt/work/test/dissonance")

text_csv = BASE / "own_script/dialogue_3/dialogue_3_vad_text.csv"
diss_csv = BASE / "own_script/dialogue_3/dialogue_3_dissonance.csv"

df_text = pd.read_csv(text_csv)
df_diss = pd.read_csv(diss_csv)

df_merged = df_text.merge(df_diss, on="utterance_id", suffixes=("_textfile", ""))
df_merged.head()


,utterance_id,text,valence_text,arousal_text,dominance_text,valence_text_n,arousal_text_n,dominance_text_n,aro_s,val_s,aro_t,val_t,delta_arousal,delta_valence,dissonant_arousal,dissonant_valence,dissonant_any
0,1,I've been feeling really overwhelmed lately. M...,2.097536,3.681398,2.381162,-0.451232,0.340699,-0.309419,0.114438,-0.637436,0.340699,-0.451232,0.226261,0.186204,False,False,False
1,2,"When I feel the palpitations, my heart starts ...",2.330700,3.645309,2.423089,-0.334650,0.322655,-0.288456,0.455570,-0.584808,0.322655,-0.334650,0.132915,0.250158,False,False,False
2,3,"Well, the last time I went to the doctor, they...",2.717745,3.235296,2.625666,-0.141128,0.117648,-0.187167,0.042801,-0.405845,0.117648,-0.141128,0.074847,0.264717,False,False,False
3,4,I guess if I could really believe that my hear...,2.622219,3.526547,2.657344,-0.188891,0.263273,-0.171328,0.149988,-0.540649,0.263273,-0.188891,0.113286,0.351758,False,False,False
4,5,"I think trying those techniques could help, bu...",2.975230,3.375428,3.043380,-0.012385,0.187714,0.021690,0.031278,-0.191446,0.187714,-0.012385,0.156436,0.179061,False,False,False


## Prompt condition 1 – Baseline therapist (ไม่เห็น VA/dissonance)

In [2]:
BASE_SYSTEM = """
You are "Luna", a warm, empathetic CBT therapist.

Your goals:
- Understand the client's thoughts, emotions, and behaviors.
- Validate their feelings without dismissing or catastrophizing.
- Gently use CBT techniques (identify automatic thoughts, examine evidence,
  explore alternative perspectives, plan small experiments).

Rules:
- Reply in a natural, conversational tone (2–4 sentences).
- Do NOT mention any numeric scores, analytics, or models.
- End most responses with an open question that invites reflection.
"""

BASE_USER_TEMPLATE = """
Client's previous message:
"{client_text}"

Please write your next therapist response to the client.
"""


In [3]:
# Method to call prompt condition 1:
example = df_merged.iloc[0]  # หรือเลือก index อื่น
user = BASE_USER_TEMPLATE.format(client_text=example["text"])
messages = [
    {"role": "system", "content": BASE_SYSTEM},
    {"role": "user", "content": user},
]


## Prompt condition 2 – Emotion-aware (เห็น text VA แต่ไม่เห็น dissonance)

In [4]:
EMOTION_SYSTEM = """
You are "Luna", a warm CBT therapist.

You receive for each client message:
- The raw text of what the client said.
- An estimated emotional profile from text analysis:
  - Valence: from -1 (very negative) to +1 (very positive)
  - Arousal: from -1 (very low/flat) to +1 (very activated/agitated)

Use this emotional information to:
- Adjust your empathy (e.g., acknowledge high distress when arousal is high and valence low).
- Choose questions that fit the emotional intensity.
- Still focus on CBT techniques (thoughts, evidence, alternative views).

Important:
- NEVER mention numbers or "valence/arousal" explicitly.
- Talk only in natural emotional language (e.g., "it sounds very overwhelming").
"""

EMOTION_USER_TEMPLATE = """
Client's previous message:
"{client_text}"

Estimated emotion from their words:
- Valence (text): {val_t:.2f}
- Arousal (text): {aro_t:.2f}

Write your next therapist response.
Remember: use this emotional profile internally to guide your tone and focus,
but do NOT mention these scores directly.
"""


In [5]:
# # Method to call prompt condition 2:
# user = EMOTION_USER_TEMPLATE.format(
#     client_text=row["text"],
#     val_t=row["valence_text_n"],
#     aro_t=row["arousal_text_n"],
# )

## Prompt condition 3 – Dissonance-aware (เห็น mismatch ระหว่าง speech vs text

In [6]:
DISSONANCE_SYSTEM = """
You are "Luna", a CBT therapist with access to both the client's words and
an analysis of how their voice matches (or mismatches) those words.

For each client message you receive:
- Text-based emotion:
  - Valence_text, Arousal_text  (from -1 to +1)
- Voice-based emotion:
  - Valence_speech, Arousal_speech (from -1 to +1)
- Dissonance:
  - delta_valence = speech - text
  - delta_arousal = speech - text

Interpretation guidelines:
- Large |delta_valence| or |delta_arousal| means the client's tone and words
  are pulling in different directions (they might be minimizing or masking something).
- Example: text seems "I'm fine" (positive) but voice is very flat or tense (negative).

Your job:
- When dissonance is small, respond as in normal emotion-aware CBT.
- When dissonance is large, gently explore the mismatch:
  - Reflect what might be "under the surface".
  - Ask curious, non-judgmental questions like
    "I wonder if part of you feels more scared/sad than the words suggest?"

Important:
- NEVER mention numbers, "dissonance", or "analysis".
- Speak only in natural language.
- Still follow CBT principles (thoughts, evidence, alternative perspectives).
"""

DISSONANCE_USER_TEMPLATE = """
Client's previous message:
"{client_text}"

Estimates from analysis:
- Text emotion:
    - Valence_text: {val_t:.2f}
    - Arousal_text: {aro_t:.2f}
- Voice emotion:
    - Valence_speech: {val_s:.2f}
    - Arousal_speech: {aro_s:.2f}
- Dissonance:
    - delta_valence (speech - text): {delta_v:.2f}
    - delta_arousal (speech - text): {delta_a:.2f}

Write your next therapist response using this information internally.
If the mismatch (absolute delta) is large, gently explore what might be
unspoken or minimized, without naming any numbers.
"""


In [7]:
# # Method to call prompt condition 3:
# user = DISSONANCE_USER_TEMPLATE.format(
#     client_text=row["text"],
#     val_t=row["val_t"],
#     aro_t=row["aro_t"],
#     val_s=row["val_s"],
#     aro_s=row["aro_s"],
#     delta_v=row["delta_valence"],
#     delta_a=row["delta_arousal"],
# )


## Save reply output

In [8]:
def call_llm(system_prompt: str, user_prompt: str) -> str:
    return query_llm(system_prompt, user_prompt)

### Call llm - GPT-4o-mini

In [9]:
from openai import OpenAI
import getpass
import os
import json

BASE_DIR = "/home/patsakornt/work/test"
DIALOGUE_ID = 3
OUT_DIR = os.path.join(BASE_DIR, "dissonance", "own_script", f"dialogue_{DIALOGUE_ID}")
os.makedirs(OUT_DIR, exist_ok=True)

print("Setting up OpenAI client...")
if "OPENAI_API_KEY" not in os.environ:
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key
client = OpenAI()
MODEL = "gpt-4o-mini"
print("✅ OpenAI client ready:", MODEL)

Setting up OpenAI client...


Enter your OpenAI API key:  ········


✅ OpenAI client ready: gpt-4o-mini


In [10]:
def call_llm(system_prompt: str, user_prompt: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=512,
    )
    return resp.choices[0].message.content


In [11]:
rows = []

for _, row in df_merged.iterrows():
    uid = int(row["utterance_id"])
    client_text = row["text"]

    # 1) baseline
    reply_base = call_llm(
        BASE_SYSTEM,
        BASE_USER_TEMPLATE.format(client_text=client_text),
    )
    rows.append({
        "utterance_id": uid,
        "client_text": client_text,
        "condition": "baseline",
        "reply": reply_base,
        "val_t": row["val_t"],
        "aro_t": row["aro_t"],
        "val_s": row["val_s"],
        "aro_s": row["aro_s"],
        "delta_valence": row["delta_valence"],
        "delta_arousal": row["delta_arousal"],
    })

    # 2) emotion-aware
    reply_emo = call_llm(
        EMOTION_SYSTEM,
        EMOTION_USER_TEMPLATE.format(
            client_text=client_text,
            val_t=row["val_t"],
            aro_t=row["aro_t"],
        ),
    )
    rows.append({
        "utterance_id": uid,
        "client_text": client_text,
        "condition": "emotion",
        "reply": reply_emo,
        "val_t": row["val_t"],
        "aro_t": row["aro_t"],
        "val_s": row["val_s"],
        "aro_s": row["aro_s"],
        "delta_valence": row["delta_valence"],
        "delta_arousal": row["delta_arousal"],
    })

    # 3) dissonance-aware
    reply_dis = call_llm(
        DISSONANCE_SYSTEM,
        DISSONANCE_USER_TEMPLATE.format(
            client_text=client_text,
            val_t=row["val_t"],
            aro_t=row["aro_t"],
            val_s=row["val_s"],
            aro_s=row["aro_s"],
            delta_v=row["delta_valence"],
            delta_a=row["delta_arousal"],
        ),
    )
    rows.append({
        "utterance_id": uid,
        "client_text": client_text,
        "condition": "dissonance",
        "reply": reply_dis,
        "val_t": row["val_t"],
        "aro_t": row["aro_t"],
        "val_s": row["val_s"],
        "aro_s": row["aro_s"],
        "delta_valence": row["delta_valence"],
        "delta_arousal": row["delta_arousal"],
    })

import pandas as pd

df_out = pd.DataFrame(rows)
df_out.to_csv("dialogue_3_replies_all_conditions.csv", index=False)
df_out.to_json("dialogue_3_replies_all_conditions.jsonl", orient="records", lines=True)
